# Mini Project: Building a Robust Part-of-Speech Tagger

## Hidden Markov Model POS Tagger

This notebook implements a Part-of-Speech tagger using a Hidden Markov Model from scratch.

The model is trained and evaluated using the Brown Corpus with the universal POS tagset.

Main techniques used:
- Maximum Likelihood Estimation
- Laplace smoothing for unseen words
- Log-space Viterbi decoding
- Word-level accuracy evaluation

# Part 1: The Coding Implementation 

Library Usage

In [ ]:
import nltk
import random
import math

## Step 1: Data Acquisition and Splitting

Use the Natural Language Toolkit (nltk) to download the Brown Corpus using the universal tagset.

In [20]:
import nltk
import random

# Download required NLTK data
nltk.download("brown")
nltk.download("universal_tagset")

from nltk.corpus import brown

[nltk_data] Downloading package brown to /home/super/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /home/super/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


### A. import and split data 

In [21]:
# TODO : Load the corpus and split it into 80% training data and 20% testing data.

# Load Brown Corpus sentences with universal POS tags
sentences = brown.tagged_sents(tagset="universal")

# Convert to a normal list so we can shuffle and split it
sentences = list(sentences)

print("Number of sentences:", len(sentences))
print("First sentence:")
print(sentences[0])

Number of sentences: 57340
First sentence:
[('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN'), ('an', 'DET'), ('investigation', 'NOUN'), ('of', 'ADP'), ("Atlanta's", 'NOUN'), ('recent', 'ADJ'), ('primary', 'NOUN'), ('election', 'NOUN'), ('produced', 'VERB'), ('``', '.'), ('no', 'DET'), ('evidence', 'NOUN'), ("''", '.'), ('that', 'ADP'), ('any', 'DET'), ('irregularities', 'NOUN'), ('took', 'VERB'), ('place', 'NOUN'), ('.', '.')]


In [22]:
# Shuffle the sentences so training and testing data are mixed fairly
random.seed(42)
random.shuffle(sentences)

# 80% for training, 20% for testing
split_index = int(0.8 * len(sentences))

train_data = sentences[:split_index]
test_data = sentences[split_index:]

print("Training sentences:", len(train_data))
print("Testing sentences:", len(test_data))

Training sentences: 45872
Testing sentences: 11468


In [23]:
# Check one training sentence
print("Example training sentence:")
print(train_data[0])

# Check sentence length
print("\nNumber of words in this sentence:", len(train_data[0]))

Example training sentence:
[('He', 'PRON'), ('let', 'VERB'), ('her', 'PRON'), ('tell', 'VERB'), ('him', 'PRON'), ('all', 'PRT'), ('about', 'ADP'), ('the', 'DET'), ('church', 'NOUN'), ('.', '.')]

Number of words in this sentence: 10


## Step 2: Parameter Estimation & Laplace Smoothing

Train your HMM by estimating the initial ($\pi$), transition ($A$), and emission ($B$) probabilities using Maximum Likelihood Estimation (MLE) based on frequency counts in your training set.

To prevent zero-probabilities when encountering unseen words during testing, apply **Laplace (Add-$\alpha$) Smoothing** to your emission matrix. The smoothed probability of emitting word $o_t$ given tag $s_i$ is:

$$P_{\text{smoothed}}(o_t|s_i) = \frac{\text{Count}(o_t \text{ tagged as } s_i) + \alpha}{\text{Count}(s_i) + \alpha|V|}$$

Where $\alpha = 1.0$ and $|V|$ is the total number of unique words in your training vocabulary. Ensure you create a specific dictionary key (e.g., `"<OOV>"`) to store the probability mass for unseen words.



In [24]:
from collections import Counter, defaultdict

# Smoothing value required by the project
alpha = 1.0

# Special token for unseen words
OOV_TOKEN = "<OOV>"

# Helper function: make words lowercase to reduce vocabulary size
def normalize_word(word):
    return word.lower()

# Count how often each tag starts a sentence
initial_counts = Counter()

# Count tag-to-tag transitions
# Example: transition_counts["DET"]["NOUN"]
transition_counts = defaultdict(Counter)

# Count tag-to-word emissions
# Example: emission_counts["NOUN"]["dog"]
emission_counts = defaultdict(Counter)

# Count total number of each tag
tag_counts = Counter()

# Store all unique words and tags
vocab = set()
tags = set()

In [25]:
# Count initial tags, transitions, and emissions from training data

for sentence in train_data:
    # Skip empty sentences, just in case
    if len(sentence) == 0:
        continue

    # Get the first tag of the sentence
    first_word, first_tag = sentence[0]
    initial_counts[first_tag] += 1

    # Loop through every word-tag pair in the sentence
    previous_tag = None

    for word, tag in sentence:
        # Normalize word
        word = normalize_word(word)

        # Save vocabulary and tag
        vocab.add(word)
        tags.add(tag)

        # Count how many times this tag appears
        tag_counts[tag] += 1

        # Count emission: tag -> word
        emission_counts[tag][word] += 1

        # Count transition: previous_tag -> current_tag
        if previous_tag is not None:
            transition_counts[previous_tag][tag] += 1

        # Update previous tag
        previous_tag = tag

# Convert sets to sorted lists for stable order
vocab = sorted(list(vocab))
tags = sorted(list(tags))

print("Number of unique words:", len(vocab))
print("Number of POS tags:", len(tags))
print("POS tags:", tags)

Number of unique words: 45153
Number of POS tags: 12
POS tags: ['.', 'ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']


In [ ]:
# Estimate initial probabilities pi

pi = {}

total_sentences = len(train_data)

for tag in tags:
    pi[tag] = initial_counts[tag] / total_sentences

print("Initial probability:")
for tag in tags:
    print(tag, ":", pi[tag])

Initial probability example:
. : 0.0870247645622602
ADJ : 0.03483606557377049
ADP : 0.12371381234740146
ADV : 0.09042553191489362
CONJ : 0.0492893268224625
DET : 0.21365974886641087
NOUN : 0.14150244157656086
NUM : 0.016437042204394837
PRON : 0.1595962678758284
PRT : 0.03712504359958144
VERB : 0.04582316009766306
X : 0.0005667945587722358


In [ ]:
# Estimate transition probabilities A

A = {}

for prev_tag in tags:
    A[prev_tag] = {}

    # Total transitions starting from prev_tag
    total_transitions_from_prev_tag = sum(transition_counts[prev_tag].values())

    for next_tag in tags:
        if total_transitions_from_prev_tag == 0:
            A[prev_tag][next_tag] = 0.0
        else:
            A[prev_tag][next_tag] = (
                transition_counts[prev_tag][next_tag] / total_transitions_from_prev_tag
            )

print("Transition probability:")
print("P(NOUN -> VERB) =", A["NOUN"]["VERB"], "(VERB|NOUN)")

Transition probability:
P(NOUN -> VERB) = 0.15922801845330714


In [32]:
# Estimate emission probabilities B using Laplace smoothing

B = {}

# We add one extra vocabulary item for OOV words
vocab_size_with_oov = len(vocab) + 1

for tag in tags:
    B[tag] = {}

    # Denominator for Laplace smoothing
    denominator = tag_counts[tag] + alpha * vocab_size_with_oov

    # Probability for each known word
    for word in vocab:
        B[tag][word] = (
            emission_counts[tag][word] + alpha
        ) / denominator

    # Probability for unseen words
    B[tag][OOV_TOKEN] = alpha / denominator

print("Emission probability:")
print("P(dog | NOUN) =", B["NOUN"].get("dog", B["NOUN"][OOV_TOKEN]))
print("P(<OOV> | NOUN) =", B["NOUN"][OOV_TOKEN])

Emission probability:
P(dog | NOUN) = 0.00020324744246968225
P(<OOV> | NOUN) = 3.763841527216338e-06


In [34]:
# Quick probability checks

print("Sum of pi probabilities:", sum(pi.values()))

example_tag = "NOUN"

print("Sum of emission probabilities for", example_tag, ":")
print(sum(B[example_tag].values()))

example_prev_tag = "NOUN"

print("Sum of transition probabilities from", example_prev_tag, ":")
print(sum(A[example_prev_tag].values()))

Sum of pi probabilities: 1.0
Sum of emission probabilities for NOUN :
1.0000000000003673
Sum of transition probabilities from NOUN :
1.0


## Step 3: The Log-Space Viterbi Decoder

Because standard probabilities are strictly between 0 and 1, multiplying them together for a normal-length sentence results in infinitesimally small numbers (Underflow). Transform the recursive Viterbi step into log-space:

$$\log v_t(j) = \max_{i=1}^N \left[ \log v_{t-1}(i) + \log P(s_j|s_i) \right] + \log P(o_t|s_j)$$

*Hint: $\log(0)$ is mathematically undefined. Add a tiny constant (epsilon, $10^{-12}$) to all probabilities before taking the logarithm to avoid math domain errors.*


In [35]:
import math

# Tiny value to avoid log(0)
EPSILON = 1e-12

def get_emission_probability(tag, word, B):
    """
    Get P(word | tag).
    If the word is unseen, use the <OOV> probability.
    """
    word = normalize_word(word)

    if word in B[tag]:
        return B[tag][word]
    else:
        return B[tag][OOV_TOKEN]


def log_viterbi(sentence, tags, A, B, pi):
    """
    Predict the best POS tag sequence for a sentence using log-space Viterbi.

    sentence: list of words
    tags: list of all possible POS tags
    A: transition probabilities
    B: emission probabilities
    pi: initial probabilities
    """

    # If sentence is empty, return empty tag list
    if len(sentence) == 0:
        return []

    # viterbi[t][tag] stores the best log-probability
    # for position t ending with this tag
    viterbi = []

    # backpointer[t][tag] stores the best previous tag
    # that leads to this current tag
    backpointer = []

    # -------------------------
    # 1. Initialization step
    # -------------------------
    first_word = sentence[0]

    viterbi.append({})
    backpointer.append({})

    for tag in tags:
        emission_prob = get_emission_probability(tag, first_word, B)

        # log P(tag starts sentence) + log P(first_word | tag)
        viterbi[0][tag] = math.log(pi.get(tag, 0.0) + EPSILON) + math.log(emission_prob + EPSILON)

        # First word has no previous tag
        backpointer[0][tag] = None

    # -------------------------
    # 2. Forward pass
    # -------------------------
    for t in range(1, len(sentence)):
        word = sentence[t]

        viterbi.append({})
        backpointer.append({})

        for current_tag in tags:
            emission_prob = get_emission_probability(current_tag, word, B)

            best_previous_tag = None
            best_log_prob = float("-inf")

            for previous_tag in tags:
                transition_prob = A[previous_tag].get(current_tag, 0.0)

                # Previous best score
                # + log P(current_tag | previous_tag)
                # + log P(word | current_tag)
                log_prob = (
                    viterbi[t - 1][previous_tag]
                    + math.log(transition_prob + EPSILON)
                    + math.log(emission_prob + EPSILON)
                )

                if log_prob > best_log_prob:
                    best_log_prob = log_prob
                    best_previous_tag = previous_tag

            viterbi[t][current_tag] = best_log_prob
            backpointer[t][current_tag] = best_previous_tag

    # -------------------------
    # 3. Backtracking step
    # -------------------------

    # Find the best final tag
    best_final_tag = max(viterbi[-1], key=viterbi[-1].get)

    best_tag_sequence = [best_final_tag]

    # Move backward from the last word to the first word
    for t in range(len(sentence) - 1, 0, -1):
        best_previous_tag = backpointer[t][best_tag_sequence[-1]]
        best_tag_sequence.append(best_previous_tag)

    # Reverse because we collected tags from end to start
    best_tag_sequence.reverse()

    return best_tag_sequence

In [36]:
# Test the Viterbi decoder on one test sentence

example_sentence_tagged = test_data[0]

# Separate words and true tags
example_words = [word for word, tag in example_sentence_tagged]
true_tags = [tag for word, tag in example_sentence_tagged]

# Predict tags
predicted_tags = log_viterbi(example_words, tags, A, B, pi)

print("Words:")
print(example_words)

print("\nTrue tags:")
print(true_tags)

print("\nPredicted tags:")
print(predicted_tags)

Words:
['According', 'to', 'state', 'law', 'a', 'slave', 'had', 'to', 'be', 'at', 'least', 'thirty', 'years', 'old', 'before', 'he', 'could', 'be', 'freed', '.']

True tags:
['ADP', 'ADP', 'NOUN', 'NOUN', 'DET', 'NOUN', 'VERB', 'PRT', 'VERB', 'ADP', 'ADJ', 'NUM', 'NOUN', 'ADJ', 'ADP', 'PRON', 'VERB', 'VERB', 'VERB', '.']

Predicted tags:
['ADP', 'ADP', 'NOUN', 'NOUN', 'DET', 'NOUN', 'VERB', 'PRT', 'VERB', 'ADP', 'ADJ', 'NUM', 'NOUN', 'ADJ', 'ADP', 'PRON', 'VERB', 'VERB', 'VERB', '.']


In [37]:
# Display word-by-word comparison

for word, true_tag, predicted_tag in zip(example_words, true_tags, predicted_tags):
    print(f"{word:15s} True: {true_tag:6s} Predicted: {predicted_tag:6s}")

According       True: ADP    Predicted: ADP   
to              True: ADP    Predicted: ADP   
state           True: NOUN   Predicted: NOUN  
law             True: NOUN   Predicted: NOUN  
a               True: DET    Predicted: DET   
slave           True: NOUN   Predicted: NOUN  
had             True: VERB   Predicted: VERB  
to              True: PRT    Predicted: PRT   
be              True: VERB   Predicted: VERB  
at              True: ADP    Predicted: ADP   
least           True: ADJ    Predicted: ADJ   
thirty          True: NUM    Predicted: NUM   
years           True: NOUN   Predicted: NOUN  
old             True: ADJ    Predicted: ADJ   
before          True: ADP    Predicted: ADP   
he              True: PRON   Predicted: PRON  
could           True: VERB   Predicted: VERB  
be              True: VERB   Predicted: VERB  
freed           True: VERB   Predicted: VERB  
.               True: .      Predicted: .     


## Step 4: Evaluation

Write an evaluation loop to process the 20% test sentences and compute your total word-level accuracy.

$$\text{Accuracy} = \frac{\text{Total Correctly Tagged Words}}{\text{Total Words in Test Set}}$$


In [38]:
# First, test on a small number of sentences to make sure the function works

small_test_data = test_data[:5]

total_words = 0
correct_words = 0

for sentence in small_test_data:
    # Separate words and true tags
    words = [word for word, tag in sentence]
    true_tags = [tag for word, tag in sentence]

    # Predict tags using our Viterbi decoder
    predicted_tags = log_viterbi(words, tags, A, B, pi)

    # Compare true tags and predicted tags
    for true_tag, predicted_tag in zip(true_tags, predicted_tags):
        total_words += 1

        if true_tag == predicted_tag:
            correct_words += 1

small_accuracy = correct_words / total_words

print("Small test total words:", total_words)
print("Small test correct words:", correct_words)
print("Small test accuracy:", small_accuracy)

Small test total words: 85
Small test correct words: 78
Small test accuracy: 0.9176470588235294


In [39]:
# Full evaluation on the whole 20% test set

total_words = 0
correct_words = 0

# Store failed sentences for later error analysis
failed_sentences = []

for sentence_index, sentence in enumerate(test_data):
    # Separate words and true tags
    words = [word for word, tag in sentence]
    true_tags = [tag for word, tag in sentence]

    # Predict tags
    predicted_tags = log_viterbi(words, tags, A, B, pi)

    # Count whether this sentence has any mistake
    sentence_has_error = False

    # Compare word by word
    for word, true_tag, predicted_tag in zip(words, true_tags, predicted_tags):
        total_words += 1

        if true_tag == predicted_tag:
            correct_words += 1
        else:
            sentence_has_error = True

    # Save failed sentence examples for report later
    if sentence_has_error:
        failed_sentences.append((words, true_tags, predicted_tags))

    # Print progress every 500 sentences
    if (sentence_index + 1) % 500 == 0:
        print("Processed sentences:", sentence_index + 1)

accuracy = correct_words / total_words

print("\nEvaluation finished!")
print("Total words:", total_words)
print("Correctly tagged words:", correct_words)
print("Accuracy:", accuracy)
print("Accuracy percentage:", accuracy * 100, "%")

Processed sentences: 500
Processed sentences: 1000
Processed sentences: 1500
Processed sentences: 2000
Processed sentences: 2500
Processed sentences: 3000
Processed sentences: 3500
Processed sentences: 4000
Processed sentences: 4500
Processed sentences: 5000
Processed sentences: 5500
Processed sentences: 6000
Processed sentences: 6500
Processed sentences: 7000
Processed sentences: 7500
Processed sentences: 8000
Processed sentences: 8500
Processed sentences: 9000
Processed sentences: 9500
Processed sentences: 10000
Processed sentences: 10500
Processed sentences: 11000

Evaluation finished!
Total words: 232177
Correctly tagged words: 217777
Accuracy: 0.9379783527222766
Accuracy percentage: 93.79783527222766 %


In [40]:
# Show some failed sentences for error analysis

num_examples = 5

for i in range(min(num_examples, len(failed_sentences))):
    words, true_tags, predicted_tags = failed_sentences[i]

    print("=" * 80)
    print("Failed sentence", i + 1)
    print("Sentence:")
    print(" ".join(words))

    print("\nWord-by-word comparison:")
    for word, true_tag, predicted_tag in zip(words, true_tags, predicted_tags):
        if true_tag != predicted_tag:
            mark = "<-- wrong"
        else:
            mark = ""

        print(f"{word:15s} True: {true_tag:6s} Predicted: {predicted_tag:6s} {mark}")

Failed sentence 1
Sentence:
With tips , the girls average between $150 and $200 a week , depending on basic salary .

Word-by-word comparison:
With            True: ADP    Predicted: ADP    
tips            True: NOUN   Predicted: NOUN   
,               True: .      Predicted: .      
the             True: DET    Predicted: DET    
girls           True: NOUN   Predicted: NOUN   
average         True: VERB   Predicted: NOUN   <-- wrong
between         True: ADP    Predicted: ADP    
$150            True: NOUN   Predicted: NOUN   
and             True: CONJ   Predicted: CONJ   
$200            True: NOUN   Predicted: ADP    <-- wrong
a               True: DET    Predicted: DET    
week            True: NOUN   Predicted: NOUN   
,               True: .      Predicted: .      
depending       True: ADP    Predicted: VERB   <-- wrong
on              True: ADP    Predicted: ADP    
basic           True: ADJ    Predicted: ADJ    
salary          True: NOUN   Predicted: NOUN   
.             

In [41]:
# Reusable evaluation function

def evaluate_model(test_data, tags, A, B, pi):
    """
    Evaluate the HMM POS tagger on test data.

    Returns:
    - accuracy
    - total_words
    - correct_words
    - failed_sentences
    - mistakes
    """

    total_words = 0
    correct_words = 0

    failed_sentences = []
    mistakes = []

    for sentence in test_data:
        words = [word for word, tag in sentence]
        true_tags = [tag for word, tag in sentence]

        predicted_tags = log_viterbi(words, tags, A, B, pi)

        sentence_has_error = False
        sentence_mistakes = []

        for word, true_tag, predicted_tag in zip(words, true_tags, predicted_tags):
            total_words += 1

            if true_tag == predicted_tag:
                correct_words += 1
            else:
                sentence_has_error = True
                mistakes.append((word, true_tag, predicted_tag))
                sentence_mistakes.append((word, true_tag, predicted_tag))

        if sentence_has_error:
            failed_sentences.append((words, true_tags, predicted_tags, sentence_mistakes))

    accuracy = correct_words / total_words

    return accuracy, total_words, correct_words, failed_sentences, mistakes

In [42]:
# Run final evaluation

accuracy, total_words, correct_words, failed_sentences, mistakes = evaluate_model(
    test_data, tags, A, B, pi
)

print("Final Evaluation Results")
print("------------------------")
print("Total words:", total_words)
print("Correct words:", correct_words)
print("Accuracy:", accuracy)
print("Accuracy percentage:", round(accuracy * 100, 2), "%")
print("Number of failed sentences:", len(failed_sentences))
print("Number of wrong word predictions:", len(mistakes))

Final Evaluation Results
------------------------
Total words: 232177
Correct words: 217777
Accuracy: 0.9379783527222766
Accuracy percentage: 93.8 %
Number of failed sentences: 7140
Number of wrong word predictions: 14400


In [43]:
# Find the most common true-tag to predicted-tag mistakes

from collections import Counter

mistake_types = Counter()

for word, true_tag, predicted_tag in mistakes:
    mistake_types[(true_tag, predicted_tag)] += 1

print("Most common POS tag mistakes:")
print("----------------------------")

for (true_tag, predicted_tag), count in mistake_types.most_common(10):
    print(f"True: {true_tag:6s}  Predicted: {predicted_tag:6s}  Count: {count}")

Most common POS tag mistakes:
----------------------------
True: VERB    Predicted: NOUN    Count: 1337
True: NOUN    Predicted: DET     Count: 1113
True: NOUN    Predicted: ADJ     Count: 986
True: NOUN    Predicted: .       Count: 709
True: PRT     Predicted: ADP     Count: 670
True: NOUN    Predicted: VERB    Count: 637
True: NOUN    Predicted: PRON    Count: 609
True: VERB    Predicted: DET     Count: 560
True: ADJ     Predicted: NOUN    Count: 558
True: VERB    Predicted: ADJ     Count: 551


In [44]:
# Show 5 failed sentence examples for report

num_examples = 5

for i in range(min(num_examples, len(failed_sentences))):
    words, true_tags, predicted_tags, sentence_mistakes = failed_sentences[i]

    print("=" * 100)
    print("Failed sentence", i + 1)
    print("Sentence:")
    print(" ".join(words))

    print("\nMistakes in this sentence:")
    for word, true_tag, predicted_tag in sentence_mistakes:
        print(f"Word: {word:15s} True: {true_tag:6s} Predicted: {predicted_tag:6s}")

    print("\nFull comparison:")
    for word, true_tag, predicted_tag in zip(words, true_tags, predicted_tags):
        mark = "<-- wrong" if true_tag != predicted_tag else ""
        print(f"{word:15s} True: {true_tag:6s} Predicted: {predicted_tag:6s} {mark}")

Failed sentence 1
Sentence:
With tips , the girls average between $150 and $200 a week , depending on basic salary .

Mistakes in this sentence:
Word: average         True: VERB   Predicted: NOUN  
Word: $200            True: NOUN   Predicted: ADP   
Word: depending       True: ADP    Predicted: VERB  

Full comparison:
With            True: ADP    Predicted: ADP    
tips            True: NOUN   Predicted: NOUN   
,               True: .      Predicted: .      
the             True: DET    Predicted: DET    
girls           True: NOUN   Predicted: NOUN   
average         True: VERB   Predicted: NOUN   <-- wrong
between         True: ADP    Predicted: ADP    
$150            True: NOUN   Predicted: NOUN   
and             True: CONJ   Predicted: CONJ   
$200            True: NOUN   Predicted: ADP    <-- wrong
a               True: DET    Predicted: DET    
week            True: NOUN   Predicted: NOUN   
,               True: .      Predicted: .      
depending       True: ADP    Predict

# OOV

In [45]:
# Check Out-of-Vocabulary words in the test set

test_total_words = 0
test_oov_words = 0
oov_examples = []

vocab_set = set(vocab)

for sentence in test_data:
    for word, tag in sentence:
        word_normalized = normalize_word(word)
        test_total_words += 1

        if word_normalized not in vocab_set:
            test_oov_words += 1

            if len(oov_examples) < 20:
                oov_examples.append(word)

oov_rate = test_oov_words / test_total_words

print("OOV Analysis")
print("------------")
print("Total test words:", test_total_words)
print("OOV words:", test_oov_words)
print("OOV rate:", round(oov_rate * 100, 2), "%")
print("Example OOV words:", oov_examples)

OOV Analysis
------------
Total test words: 232177
OOV words: 5050
OOV rate: 2.18 %
Example OOV words: ['non-partisan', 'chi-chi', 'cosy', 'chaperon', 'Pursewarden', "Durrell's", "Stober's", '13-16', 'beardown', '18,792', 'pickle', 'mayonnaise', 'extras', 'tangy', 'chive', 'conversant', 'deadlines', 'instalments', 'Puppies', 'Antinomians']
